### Importar librerías cirq y qsimcirq

In [1]:
try:
    import cirq
except ImportError:
    !pip install cirq --quiet
    import cirq

try:
    import qsimcirq
except ImportError:
    !pip install qsimcirq --quiet
    import qsimcirq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.3/580.3 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 22.0 MB/s eta 0:00:00


In [19]:
import cirq
import numpy as np
from scipy.optimize import minimize

n = 12
qubits = cirq.LineQubit.range(n)
J = 1.0
h = 0.5


p = 6

In [20]:
def build_qaoa_circuit(gammas, betas):
    circuit = cirq.Circuit()

    # Estado inicial |+>
    circuit.append(cirq.H.on_each(*qubits))

    for layer in range(p):
        gamma = gammas[layer]
        beta = betas[layer]

        # --- Cost Hamiltonian (Ising 1D) ---
        for i in range(n - 1):
            circuit.append(cirq.ZZ(qubits[i], qubits[i+1]) ** (gamma * J / np.pi))

        for i in range(n):
            circuit.append(cirq.Z(qubits[i]) ** (gamma * h / np.pi))

        # --- Mixer ---
        for i in range(n):
            circuit.append(cirq.X(qubits[i]) ** (beta / np.pi))

    return circuit

In [21]:
sim = qsimcirq.QSimSimulator()

def compute_energy(state_vector):
    energy = 0.0

    # ZZ terms
    for i in range(n - 1):
        for basis_state, amp in enumerate(state_vector):
            prob = np.abs(amp)**2

            zi = 1 if ((basis_state >> i) & 1) == 0 else -1
            zj = 1 if ((basis_state >> (i+1)) & 1) == 0 else -1

            energy += J * zi * zj * prob

    # Z terms
    for i in range(n):
        for basis_state, amp in enumerate(state_vector):
            prob = np.abs(amp)**2

            zi = 1 if ((basis_state >> i) & 1) == 0 else -1

            energy += h * zi * prob

    return energy

In [22]:
def objective(params):
    gammas = params[:p]
    betas = params[p:]

    circuit = build_qaoa_circuit(gammas, betas)
    result = sim.simulate(circuit)

    state = result.final_state_vector

    return compute_energy(state)

In [23]:
# inicialización aleatoria, se puede cambiar para ver mejora
init_params = np.random.uniform(0, np.pi, 2*p)

res = minimize(
    objective,
    init_params,
    method='COBYLA',   # robusto para QAOA
    options={'maxiter': 100}
)

print("Energía mínima encontrada:", res.fun)
print("Parámetros óptimos:", res.x)

Energía mínima encontrada: -7.589593410491943
Parámetros óptimos: [2.12896631 2.92676848 3.27091392 1.11914508 2.8093797  1.2685559
 2.48346562 0.78068629 2.39031183 2.94354553 0.39536903 0.89826472]
